# 업그레이드 방향

지금 구조는 이미 꽤 좋은 기본형 RAG 파이프라인이고, 다음 업그레이드는 “검색을 더 많이 붙이는 것”보다 검색된 근거가 정말 답변하기에 충분한지 판단하는 단계를 넣는 쪽이 효과가 크다.

[pipeline_diagram](3.%20pipeline_diagram.png)

현재
```
Question
  ↓
Hybrid Retrieval
  ↓
Reranker
  ↓
Generate
  ↓
Citation
  ↓
Grounding
  ↓
Query Rewrite
```

업그레이드
```
Question
  ↓
Query Analysis ***
  ↓
Hybrid Retrieval
  ↓
Reranker
  ↓
Context Sufficiency ***
  ↓
Evidence Selection ***
  ↓
Sentence Plan ***
  ↓
Generate
  ↓
Claim-level Citation
  ↓
Verifier
  ↓
PASS ─────→ Final Answer
  │
  FAIL
  ↓
Failure-aware Rewrite
  ↓
재검색 / 재생성
```

### 가장 추천하는 업그레이드 순위
1. **Context Sufficiency** : 이 근거들만으로 질문에 답할 수 있는가? 판단
2. **Evidence Selection** : reranker에서 나온 모든 문장이 답변 필요한건 아님. (LLM에게 들어가는 노이즈 줄이기 용)
3. **Claim-level Grounding** : 지금 grounding_check()은 문장 단위로 검증 -> 답변을 atomic claim으로 쪼개 검증
4. **Sentence Plan** : 답변을 바로 생성하는게 아니라, 무슨 사실을 말할지 + 어떤 근거를 쓸지 먼저 계획.
    ```
    LLM이 일단 글 씀 -> 나중에 Citation 붙이기
    가 아니라 
    Evidence -> Claim -> Sentence
    순서로
    ```
5. **Failure-aware Query Rewrite** : 현재 Grounding 실패하면 그냥 query rewrite 하는데, 왜 실패했는가를 먼저 분류한 후 그 다음 전략을 다르게 설정
    ```
    MISSING_EVIDENCE
    → missing claim 중심으로 검색

    WRONG_DOCUMENT
    → metadata / keyword 강화

    NUMERIC_CONFLICT
    → 숫자 + 단위 + 기간 포함해서 검색

    AMBIGUOUS_QUERY
    → query decomposition

    CONTRADICTORY_EVIDENCE
    → 추가 문서 검색
    ```
6. **Query Decomposition** : 복잡한 질문에서 특히 필요. 아래처럼 질문 분해 후 나중에 evidence merge
    ```
    도서관 운영시간이랑 대출 기간,
    연체하면 어떻게 되는지도 알려줘.
    ->
    Q1. 도서관 운영시간
    Q2. 도서 대출 기간
    Q3. 연체 시 규정
    ```
7. **Retrieval Evaluation** : test_cases를 만들고 Recall@K, MRR, Hit Rate, NDCG 측정. -> BM25 추가가 도움이 됐는지, Reranker가 도움이 됐는지, chunk overlap이 좋은지 측정 가능

- **Grounding 결과를 2개가 아니라 3단계로** : PASS, FAIL 보단 CONFIRMED,PARTIAL, UNKNOWN
- **Parent-Child Chunking** : 현재는 chunk_size가 고정인데, 고급 구조에선 검색용 chunk는 작게, LLM Context용 chunk는 크게 설정.